# Webinar-Ready: SQL Joins with SQLite Magic (90 minutes) — insurance.db

**Goal:** teach SQL joins (basic → advanced) using a real dataset, with hands-on practice.  
**SQL runs inside the notebook** using `%%sql` magic (SQLite).

## What learners need (before we start)
1. VS Code extensions: **Jupyter**
2. Python packages:
   ```bash
   pip install ipython-sql sqlalchemy
   ```
3. Files in the same folder as this notebook:
   - `insurance.db`  *(created by `convert_insurance_to_db.py`)*

---

##  Webinar Flow

| Segment | Output |
|---|---|
| Setup & connect to DB | Everyone can run `%%sql` |
 Assessment (attempt only) | Learners try Q1–Q4 |
 Theory recap + join intuition | Clear mental model |
| Basic joins: INNER + LEFT | Read outputs correctly |
| Intermediate: RIGHT/FULL OUTER (SQLite emulation) | Demo NULLs/orphans |
 Advanced: CROSS JOIN + SELF JOIN | Grids + pair matching |
| Advanced: window functions patterns | Ranking / percentiles |
| Reveal solutions + wrap-up | Debrief |


In [1]:
import os
from pathlib import Path

# --- Project paths (adjust if you moved the folder) ---
PROJECT_DIR = Path.home() / 'Documents' / 'SQL_Joins'
DB_PATH = PROJECT_DIR / 'insurance.db'

# Make notebook run from the project folder (so relative paths work)
os.chdir(PROJECT_DIR)

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Could not find insurance.db at: {DB_PATH}\n"
        "Create it first by running convert_insurance_to_db.py (CSV → DB)."
    )

# Load SQL magic and connect (use absolute path)
%load_ext sql
%config SqlMagic.autopandas = True

conn_str = f"sqlite:////{DB_PATH}"
%sql $conn_str

print('Connected to:', DB_PATH)


Connected to: /Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db


#  SQL Joins Theory 



## Why joins exist
- Databases split data into multiple tables to reduce duplication and keep it consistent.
- A **JOIN** combines rows from two tables using a condition (usually matching keys).

## Keys and grain (most important concept)
- **Primary key (PK):** uniquely identifies a row.
- **Foreign key (FK):** points to a row in another table.
- **Grain:** what a row represents (one person? one claim? one payment?).  
If grain is unclear, joins often create duplicates.

## INNER JOIN
- Returns **only rows that match** on both sides.

## LEFT JOIN
- Returns **all rows from the left** table.
- Non-matches on the right become **NULL**.

## RIGHT JOIN (SQLite note)
- SQLite doesn’t support RIGHT JOIN.
- Emulate by swapping tables and using LEFT JOIN.

## FULL OUTER JOIN (SQLite note)
- SQLite doesn’t support FULL OUTER JOIN.
- Emulate using `UNION ALL` of two LEFT JOINs (plus a filter for the unmatched side).

## CROSS JOIN
- All combinations (Cartesian product).
- Great for reporting grids (e.g., region × smoker).

## SELF JOIN
- A table joined to itself.
- Useful for comparisons (similarity, duplicates, hierarchies).

## Pitfalls
- Wrong key ⇒ duplicates (accidental many-to-many).
- NULL never matches `=`.
- Always sanity-check row counts.


## 0) Setup (0–5 min)
Run these two cells first.

In [2]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


 **Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

In [3]:
%reload_ext sql

**Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

In [4]:
import os
print(os.getcwd())
print(os.path.exists("insurance.db"))

/Users/nhlakaniphokwazimthembu/Documents/SQL_Joins
True


In [5]:
%%sql
SELECT name, type
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY type, name;

 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,name,type
0,dim_region,table
1,dim_sex,table
2,dim_smoker,table
3,insurance_fact,table
4,person,table
5,sqlite_sequence,table
6,vw_cross_join_region_smoker_grid,view
7,vw_fact_filtered_demo,view
8,vw_fact_with_orphans_demo,view
9,vw_full_outer_join_demo,view


In [6]:
import os
os.chdir("/Users/nhlakaniphokwazimthembu/Documents/SQL_Joins")
os.getcwd()

'/Users/nhlakaniphokwazimthembu/Documents/SQL_Joins'

 **Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

In [4]:
#import os
#os.chdir("/Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/SQ_Joins")
#os.getcwd()

 **Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

 **Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

In [7]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


# Assessment (Attempt First) — 5–15 min

**Rules for this section:**
- Try each question **without** looking at the solution.
- After time is up, we’ll reveal solutions together.

> Tip: If you get stuck, start from `vw_insurance_flat`.


**Facilitator note:** This version has **no solutions** included. Use your own answer key during the debrief.

## Q1 (Beginner): Average charges by smoker status
Return:
- smoker
- number of people
- average charges

**Expected:** smokers have higher average charges.


⚠️ **Connection note**: This notebook now uses the **single setup cell near the top** to set the working folder and connect to `insurance.db`. You can ignore/remove older connection cells.

In [11]:
%%sql
SELECT smoker,
       COUNT(*) AS n_people,
       ROUND(AVG(charges), 2) AS avg_charges
FROM vw_insurance_flat
GROUP BY smoker
ORDER BY avg_charges DESC;

 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,smoker,n_people,avg_charges
0,yes,274,32050.23
1,no,1064,8434.27


## Q2 (Intermediate): Average charges by region × smoker
Return:
- region
- smoker
- number of people
- average charges

**Expected:** 8 rows (4 regions × 2 smoker statuses).


In [12]:
%%sql
SELECT region,
        smoker,
        COUNT(*) AS n_people,
        ROUND(AVG(charges), 2) AS avg_charges
FROM vw_insurance_flat
GROUP BY region, smoker
ORDER BY region, avg_charges DESC;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,region,smoker,n_people,avg_charges
0,northeast,yes,67,29673.54
1,northeast,no,257,9165.53
2,northwest,yes,58,30192.00
3,northwest,no,267,8556.46
4,southeast,yes,91,34845.00
5,southeast,no,273,8032.22
6,southwest,yes,58,32269.06
7,southwest,no,267,8019.28


## Q3 (Intermediate → Advanced): Top 5 charges per region
Return the top 5 charges **within each region**, including:
- person_id, region, smoker, bmi, charges

**Hint:** window function `ROW_NUMBER()`.


In [13]:
%%sql
-- Write your answer here
WITH ranked AS (
    SELECT 
        person_id, region, smoker, bmi, charges,
        ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn
    FROM vw_insurance_flat  
)
SELECT *
FROM ranked
WHERE rn <= 5
ORDER BY region, rn;

 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,region,smoker,bmi,charges,rn
0,578,northeast,yes,38.095,58571.07448,1
1,282,northeast,yes,40.565,48549.17835,2
2,331,northeast,yes,36.385,48517.56315,3
3,289,northeast,yes,36.765,47896.79135,4
4,884,northeast,yes,37.050,46255.11250,5
5,1231,northwest,yes,34.485,60021.39897,1
6,820,northwest,yes,35.530,55135.40209,2
7,56,northwest,yes,36.955,47496.49445,3
8,1302,northwest,yes,30.875,46718.16325,4
9,1123,northwest,yes,36.860,46661.44240,5


## Q4 (Advanced): Top 10% charges within each region (deciles)
Mark the top 10% highest charges **per region** and summarise by region × smoker:
- n_people
- n_top_10pct
- avg_top_10pct_charges

**Hint:** `NTILE(10)`.


In [18]:
%%sql
-- Write your answer here
WITH buckets AS (
    SELECT
        region, smoker, charges,
        NTILE(10) OVER (PARTITION BY region ORDER BY charges DESC) AS decile
    FROM vw_insurance_flat
)
SELECT 
    region,
    smoker,
    COUNT(*) AS n_people,
    ROUND(AVG(CASE WHEN decile = 10 THEN charges END), 2) AS avg_top_decile_charges
FROM buckets
GROUP BY region, smoker
ORDER BY region, smoker;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,region,smoker,n_people,avg_top_decile_charges
0,northeast,no,257,2151.31
1,northeast,yes,67,NaN
2,northwest,no,267,2025.11
3,northwest,yes,58,NaN
4,southeast,no,273,1522.67
5,southeast,yes,91,NaN
6,southwest,no,267,1668.57
7,southwest,yes,58,NaN


# Theory Recap (15–25 min)

## Join cheat-sheet
- **INNER JOIN**: only matches
- **LEFT JOIN**: keep left rows; right becomes NULL when missing
- **RIGHT JOIN**: not supported in SQLite → emulate by swapping tables + LEFT JOIN
- **FULL OUTER JOIN**: not supported in SQLite → emulate with two LEFT JOINs + `UNION ALL`
- **CROSS JOIN**: all combinations (Cartesian product)
- **SELF JOIN**: table joined to itself

## Practical checks
- Confirm table grain
- Check row counts before/after joins
- Look for duplicates (accidental many-to-many)


## Baseline view (2 min)
This view matches the original CSV shape.

In [19]:
%%sql
SELECT * FROM vw_insurance_flat
LIMIT 10;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,sex,bmi,children,smoker,region,charges
0,1,19,female,27.900,0,yes,southwest,16884.92400
1,2,18,male,33.770,1,no,southeast,1725.55230
2,3,28,male,33.000,3,no,southeast,4449.46200
3,4,33,male,22.705,0,no,northwest,21984.47061
4,5,32,male,28.880,0,no,northwest,3866.85520
5,6,31,female,25.740,0,no,southeast,3756.62160
6,7,46,female,33.440,1,no,southeast,8240.58960
7,8,37,female,27.740,3,no,northwest,7281.50560
8,9,37,male,29.830,2,no,northeast,6406.41070
9,10,60,female,25.840,0,no,northwest,28923.13692


# Basic Joins (25–40 min)

## 1) INNER JOIN (10 min)
We join person + dimensions + fact (charges) into one row per person.


In [20]:
%%sql
SELECT
  p.person_id, p.age, s.sex, p.bmi, p.children, sm.smoker, rg.region, f.charges
FROM person p
JOIN insurance_fact f ON f.person_id = p.person_id
JOIN dim_sex s        ON s.sex_id = p.sex_id
JOIN dim_smoker sm    ON sm.smoker_id = p.smoker_id
JOIN dim_region rg    ON rg.region_id = p.region_id
LIMIT 10;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,sex,bmi,children,smoker,region,charges
0,1,19,female,27.900,0,yes,southwest,16884.92400
1,2,18,male,33.770,1,no,southeast,1725.55230
2,3,28,male,33.000,3,no,southeast,4449.46200
3,4,33,male,22.705,0,no,northwest,21984.47061
4,5,32,male,28.880,0,no,northwest,3866.85520
5,6,31,female,25.740,0,no,southeast,3756.62160
6,7,46,female,33.440,1,no,southeast,8240.58960
7,8,37,female,27.740,3,no,northwest,7281.50560
8,9,37,male,29.830,2,no,northeast,6406.41070
9,10,60,female,25.840,0,no,northwest,28923.13692


## 2) LEFT JOIN (5 min)
Keep all people, even if charges are missing (concept).  
In the raw dataset, everyone has charges, so it looks similar to INNER JOIN.


In [21]:
%%sql
SELECT p.person_id, p.age, f.charges
FROM person p
LEFT JOIN insurance_fact f ON f.person_id = p.person_id
LIMIT 10;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,charges
0,1,19,16884.92400
1,2,18,1725.55230
2,3,28,4449.46200
3,4,33,21984.47061
4,5,32,3866.85520
5,6,31,3756.62160
6,7,46,8240.58960
7,8,37,7281.50560
8,9,37,6406.41070
9,10,60,28923.13692


# Intermediate Joins (40–55 min)

SQLite doesn’t support RIGHT/FULL OUTER joins directly. We show **emulation patterns** and use the demo views in the DB to make differences obvious.


## RIGHT JOIN emulation (5 min): swap + LEFT JOIN

In [22]:
%%sql
SELECT f.person_id, p.age, f.charges
FROM insurance_fact f
LEFT JOIN person p ON p.person_id = f.person_id
LIMIT 10;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,charges
0,1,19,16884.92400
1,2,18,1725.55230
2,3,28,4449.46200
3,4,33,21984.47061
4,5,32,3866.85520
5,6,31,3756.62160
6,7,46,8240.58960
7,8,37,7281.50560
8,9,37,6406.41070
9,10,60,28923.13692


## FULL OUTER JOIN emulation (5 min): two LEFT JOINs + UNION ALL

In [23]:
%%sql
SELECT p.person_id, f.person_id AS fact_person_id, p.age, f.charges
FROM person p
LEFT JOIN insurance_fact f ON f.person_id = p.person_id

UNION ALL

SELECT p.person_id, f.person_id AS fact_person_id, p.age, f.charges
FROM insurance_fact f
LEFT JOIN person p ON p.person_id = f.person_id
WHERE p.person_id IS NULL;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,fact_person_id,age,charges
0,1,1,19,16884.92400
1,2,2,18,1725.55230
2,3,3,28,4449.46200
3,4,4,33,21984.47061
4,5,5,32,3866.85520
...,...,...,...,...
1333,1334,1334,50,10600.54830
1334,1335,1335,18,2205.98080
1335,1336,1336,18,1629.83350
1336,1337,1337,21,2007.94500


## Demo views (5 min): see NULLs and orphans
These views are already in `insurance.db` to make the join differences visible.


In [24]:
%%sql
SELECT 'INNER' AS join_type, COUNT(*) AS rows FROM vw_inner_join_demo
UNION ALL
SELECT 'LEFT', COUNT(*) FROM vw_left_join_demo
UNION ALL
SELECT 'RIGHT (emulated)', COUNT(*) FROM vw_right_join_demo
UNION ALL
SELECT 'FULL OUTER (emulated)', COUNT(*) FROM vw_full_outer_join_demo;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,join_type,rows
0,INNER,1205
1,LEFT,1338
2,RIGHT (emulated),1343
3,FULL OUTER (emulated),1343


In [25]:
%%sql
-- LEFT JOIN demo: where charges are missing
SELECT * FROM vw_left_join_demo
WHERE charges IS NULL
LIMIT 20;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,charges
0,10,60,None
1,20,30,None
2,30,31,None
3,40,60,None
4,50,36,None
5,60,34,None
6,70,28,None
7,80,41,None
8,90,55,None
9,100,38,None


In [26]:
%%sql
-- RIGHT JOIN demo: orphan facts (no matching person)
SELECT * FROM vw_right_join_demo
WHERE age IS NULL
ORDER BY person_id;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,age,charges
0,1339,None,9999.99
1,1340,None,8888.88
2,1341,None,7777.77
3,1342,None,6666.66
4,1343,None,5555.55


# Advanced Joins (55–70 min)

## CROSS JOIN (8 min)
We generate a region × smoker grid and LEFT JOIN stats onto it.


In [27]:
%%sql
SELECT * FROM vw_cross_join_region_smoker_grid
ORDER BY region, smoker;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,region,smoker,n_people,avg_charges
0,northeast,no,257,9165.53
1,northeast,yes,67,29673.54
2,northwest,no,267,8556.46
3,northwest,yes,58,30192.00
4,southeast,no,273,8032.22
5,southeast,yes,91,34845.00
6,southwest,no,267,8019.28
7,southwest,yes,58,32269.06


## SELF JOIN (7 min)
Find pairs of people in the same region with similar BMI.


In [28]:
%%sql
SELECT * FROM vw_self_join_similar_bmi_pairs
ORDER BY bmi_diff ASC, region
LIMIT 50;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_a,person_b,region,bmi_a,bmi_b,bmi_diff
0,9,93,northeast,29.830,29.830,0.0
1,9,1053,northeast,29.830,29.830,0.0
2,9,1092,northeast,29.830,29.830,0.0
3,9,1251,northeast,29.830,29.830,0.0
4,17,366,northeast,30.780,30.780,0.0
5,17,646,northeast,30.780,30.780,0.0
6,17,829,northeast,30.780,30.780,0.0
7,17,851,northeast,30.780,30.780,0.0
8,18,855,northeast,23.845,23.845,0.0
9,24,1335,northeast,31.920,31.920,0.0


# Advanced Patterns (70–85 min)

Window functions are common in analytics:
- ranking within groups
- percentiles / deciles


In [29]:
%%sql
-- Top 3 charges per region
WITH ranked AS (
  SELECT
    person_id, region, smoker, charges,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn
  FROM vw_insurance_flat
)
SELECT *
FROM ranked
WHERE rn <= 3
ORDER BY region, rn;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,person_id,region,smoker,charges,rn
0,578,northeast,yes,58571.07448,1
1,282,northeast,yes,48549.17835,2
2,331,northeast,yes,48517.56315,3
3,1231,northwest,yes,60021.39897,1
4,820,northwest,yes,55135.40209,2
5,56,northwest,yes,47496.49445,3
6,544,southeast,yes,63770.42801,1
7,1301,southeast,yes,62592.87309,2
8,1242,southeast,yes,49577.66240,3
9,1147,southwest,yes,52590.82939,1


In [30]:
%%sql
-- Deciles within region (inspect distribution)
SELECT region,
       NTILE(10) OVER (PARTITION BY region ORDER BY charges) AS decile,
       charges
FROM vw_insurance_flat
LIMIT 30;


 * sqlite://///Users/nhlakaniphokwazimthembu/Documents/SQL_Joins/insurance.db
Done.


,region,decile,charges
0,northeast,1,1694.79640
1,northeast,1,1702.45530
2,northeast,1,1704.56810
3,northeast,1,1704.70015
4,northeast,1,1705.62450
5,northeast,1,1708.00140
6,northeast,1,1708.92575
7,northeast,1,1712.22700
8,northeast,1,1967.02270
9,northeast,1,1984.45330


# Wrap-up (85–90 min)

- Revisit the four assessment questions and show solutions (already hidden above).
- Ask: which join would you use in a real data pipeline and why?
- Homework: build BMI bands and compare charges by band × smoker × region.
